# Quantum Teleporation -- 2 Node

In this demo, we'll demonstrate the simulation of the teleportation application. The network topology has two nodes: Alice and Bob. Alice will teleport a qubit to Bob. 

## 1. The Basics

The figure below shows the quantum circuit of teleportation, as well as showing which qubits belong to Alice and which qubit belongs to Bob. We need to turn this quantum circuit into a network protocol.

<img src="./figures/teleport-circuit.png" width="600">

**Set up**: Assume node Alice and Bob both have two quantum memories, one labeled as data memory and the other labeled as communication memory. There is a quantum channel between Alice and Bob, and the BSM node is in the middle of the quantum channel.

Let's break down quantum teleportation into 4 steps.

**Step 1**: The tradition is to let Alice send a qubit to Bob. So, Alice prepares $\ket{\psi}$ at memory $d_1$.

<img src="./figures/teleport-1.png" width="600">

**Step 2**: Generate an entangled pair between Alice's memory $c_1$ and Bob's memory $c_2$.

<img src="./figures/teleport-2.png" width="600">

**Step 3**: Alice does a Bell measurement on it's memory $d_1$ and $c_1$. The measurement results are sent to Bob.

<img src="./figures/teleport-3.png" width="600">

**Step 4**: Bob does the correction on $c_2$ upon receiving the measurement results. Then $\ket{\psi}$ is teleported to Bob's memory $c_2$.

<img src="./figures/teleport-4.png" width="600">

## 2. The Code

**The imports**

- `log`: defines the behavior for the SeQUeNCe logging system
- `TeleportApp`: TeleportApp is a specialized RequestApp that implements quantum teleportation. It handles the teleportation protocol between two quantum nodes (Alice and Bob).
- `DQCNode`: Node that supports Distributed Quantum Computing
- `DQCNetTopo`: Class for generating a distributed quantum computing network with distributed quantum computing nodes
- `random_state`: Function to generate a random quantum state
- `verify_same_state_vector`: Function to verify whether two state vectors are the same or not


In [6]:
import numpy as np
import sequence.utils.log as log
from sequence.app.teleport_app import TeleportApp
from sequence.topology.node import DQCNode
from sequence.topology.dqc_net_topo import DQCNetTopo
from sequence.kernel.quantum_utils import random_state, verify_same_state_vector
from sequence.constants import MILLISECOND
import sys

### 2.1 The teleport() function

Teleport a $\ket{\psi}$ from Alice to Bob

In [7]:
def teleport(psi: np.array, seed: int = 0) -> np.array:
    """Do a teleportation
    
    Args:
        psi (np.array): the state to be teleported at Alice
        seed (int): the random seed
    Returns:
        np.array: the state received at Bob
    """
    # read the json config file, initialize the topology
    

    topo = DQCNetTopo("teleport_2node.json")
    tl   = topo.tl  # timeline

    # set up log
    log_filename = 'teleport_2node.log'

    log.set_logger(__name__, tl, log_filename)
    log.set_logger_level('DEBUG')

    log.track_module('generation')
    log.track_module('teleportation')
    log.track_module('teleport_app')

    # get alice and bob node
    nodes = topo.nodes[DQCNetTopo.DQC_NODE]
    alice = next(n for n in nodes if n.name=="alice")
    bob   = next(n for n in nodes if n.name=="bob")

    # 0) set seed for the bsm node
    bsm_nodes = topo.nodes.get(DQCNetTopo.BSM_NODE, [])
    for bsm_node in bsm_nodes:
        bsm_node.set_seed(seed)
    
    # 1) Prepare |ψ> in Alice’s data memory
    data_memo_arr = alice.get_component_by_name(alice.data_memo_arr_name)
    data_memo_arr[0].update_state(psi)

    # 2) Attach the TeleportApp on both nodes
    alice_app = TeleportApp(alice)
    bob_app   = TeleportApp(bob)

    # 3) Kick off teleport
    alice_app.start(
        responder   = bob.name,
        start_t     = 1  * MILLISECOND,
        end_t       = 100 * MILLISECOND,
        memory_size = 1,
        fidelity    = 0.8,
        data_memory_index = 0
    )
    
    # 4) Run the simulation
    tl.init()
    tl.run()

    # 5) simulation finished, read out Bob’s communication qubit state
    print("\n===== TELEPORTATION TEST FINISHED =====")
    print("Alice results:", alice_app.results)
    print("Bob results  :", bob_app.results)
    print("=======================================\n")
    routing = alice.network_manager.get_routing_protocol()

    print("Routing class:")
    print(type(routing))

    print("Routing class name:")
    print(type(routing).__name__)

    print("Routing object:")
    print(routing)
    return None

### 2.2 The main() function

In [8]:
def main():

    psi = np.array(random_state())
    seed = np.random.randint(0, 1000)

    # Run teleportation
    teleport(psi, seed)

    print(f'\nOriginal psi = {psi}')

    # Read the automatically generated log file
    with open("teleport_2node.log", "r", encoding="utf-8", errors="replace") as f:
        logs = f.read()

    # Save a clean copy
    with open("clean_log.txt", "w", encoding="utf-8") as f:
        f.write(logs)

    print("Log saved to clean_log.txt")


In [9]:
# please run this cell repeatedly
main()


alice -> (0, 0, 0)
bob -> (1, 0, 0)
Table      : {'bob': 'bob'}
Message    : |type=RSVPMsgType.REQUEST; reservation=|identity=0; initiator=alice; responder=bob; path=[]; start_time=1,000,000,000; end_time=100,000,000,000; target_fidelity=0.8; entanglement_number=1; memory_size=1||
[ROUTING] Complete path: alice -> bob
[ROUTING] Next hop: alice -> bob

===== TOPOLOGY ROUTING =====
Routing Protocol : XYRoutingProtocol
Current Node     : alice
Destination      : bob
Next Hop         : bob
Forwarding alice -> bob
Future time = 500000000
Receiver    = bob
Process     = <sequence.kernel.process.Process object at 0x000001EF67BCB930>
Event       = <sequence.kernel.event.Event object at 0x000001EF0A262CB0>
SCHEDULE: str receive_message 500000000
Event scheduled successfully
INIT = XYRoutingProtocol

===== XY ROUTING INIT =====
Current Node : alice

INIT = XYRoutingProtocol

===== XY ROUTING INIT =====
Current Node : bob

RUNNING: DQCNode receive_message 500000000

===== DQCNode.receive_message 

In [10]:
import os

print("Current directory:", os.getcwd())

print("\nteleport_2node.log exists:",
      os.path.exists("teleport_2node.log"))

if os.path.exists("teleport_2node.log"):
    print("File size:",
          os.path.getsize("teleport_2node.log"),
          "bytes")

    with open("teleport_2node.log", "r",
              encoding="utf-8", errors="replace") as f:
        content = f.read()

    print("\n===== LOG CONTENT =====")
    print(content)
    print("======================")

Current directory: c:\predoc\SeQUeNCe\example\demo_for_beginners

teleport_2node.log exists: True
File size: 979 bytes

===== LOG CONTENT =====
0                    DEBUG   teleport_app           alice.TeleportApp: initialized
0                    DEBUG   teleport_app           bob.TeleportApp: initialized
0                    DEBUG   teleport_app           alice.TeleportApp: start() -> responder=bob, data_memory_index=0
0                    DEBUG   teleportation          alice.teleport.bob.DataMemoryArray[0]: initialized (data_memory=0)
1,000,000,000        DEBUG   teleport_app           alice.TeleportApp: reservation_result -> True
100,000,000,000      DEBUG   teleport_app           bob.TeleportApp: get_memory, name=bob.MemoryArray[0], state=RAW
100,000,000,000      DEBUG   teleport_app           bob.TeleportApp: get_memory, name=bob.MemoryArray[0], state=RAW
100,000,000,000      DEBUG   teleport_app           alice.TeleportApp: get_memory, name=alice.MemoryArray[0], state=RAW
100,00

# 3 Analyze the log

<img src="./figures/teleport-timeline.png" width="800">